# PS4 - Problem 1 - Neural Networks: MNIST image classification

### (a)

We observe that the training procedure in the provided code contains a `gradient_descent_batch` function, which iterates on each sample in the batch, calling the `backward_prop_function` to process it. This function, therefore, calls the specific backpropagation functions of each operation on a single sample, instead of processing at once the whole batch. Consequently, we derive in this part the gradient computation formulae to process a single sample. 

We start by transforming the architecture presented verbally in the problem into a more mathematical version. 

#### First layer - Convolution
Assume the input sample is $X \in \R^{D_X\times D_X}$, where in this specific case $D_X = 28$. The first layer implements a convolution operation, whose kernel size of $d = 4$, and which produces 2 output channels, namely $Z^{[c]} \in \R^{D_Z\times D_Z}$ with $c \in \{1,2\}$. In this context, the layer parameters are $W_1^{[c]} \in \R^{4 \times 4}$ and $b_1^{[c]} \in \R$. More generally, one could write that $W_1 \in \R^{2 \times 4 \times 4}$ and $b_1 \in \R^2$. In addition, assuming a stride-1 convolution, the output dimension is $D_Z = D_X - d + 1 = 25$ in this specific case. This formulation allows to write:
$$
Z_{i,j}^{[c]} = b_1^{[c]} + \sum_{u = 1}^{d} \sum_{v = 1}^{d} W_{1,u,v}^{[c]} X_{i+u, j+v} 
$$

This implies that a generic matrix $Z^{[c]}$ can be written as:
$$
Z^{[c]} =
\begin{bmatrix}
W_{1,1}^{[c]} X_{1,1} + \cdots + W_{d,d}^{[c]} X_{d,d} & \cdots & W_{1,1}^{[c]} X_{1,D_Z} + \cdots + W_{d,d}^{[c]} X_{d,D_X} \\
W_{1,1}^{[c]} X_{2,1} + \cdots + W_{d,d}^{[c]} X_{d+1,d} & \cdots & W_{1,1}^{[c]} X_{2,D_Z} + \cdots + W_{d,d}^{[c]} X_{d+1,D_X} \\
\vdots & \ddots & \vdots \\
W_{1,1}^{[c]} X_{D_Z,1} + \cdots + W_{d,d}^{[c]} X_{D_X,d} & \cdots &  W_{1,1}^{[c]} X_{D_Z,D_Z} + \cdots + W_{d,d}^{[c]} X_{D_X,D_X}\\
\end{bmatrix}
+ b_1^{[c]}
$$

where $W_1^{[c]}$ was temporarily renamed to $W^{[c]}$ to simplify the notation. It holds, therefore:
$$
Z^{[c]} =
W_{1,1}^{[c]}
\begin{bmatrix}
X_{1,1} & \cdots & X_{1,D_Z} \\
X_{2,1} & \cdots & X_{2,D_Z} \\
\vdots & \ddots & \vdots \\
X_{D_Z,1} & \cdots & X_{D_Z,D_Z} \\
\end{bmatrix}
+ \cdots + 
W_{d,d}^{[c]}
\begin{bmatrix}
X_{d,d} & \cdots & X_{d,D_X} \\
X_{d+1,d} & \cdots & X_{d+1,D_X} \\
\vdots & \ddots & \vdots \\
X_{D_X,d} & \cdots & X_{D_X,D_X} \\
\end{bmatrix}
+ b_1^{[c]}
$$
$$
Z^{[c]} = b_1^{[c]} + \sum_{u = 1}^{d} \sum_{v = 1}^{d} W_{u,v}^{[c]} \tilde{X}_{u,v} 
$$

where $\tilde{X}_{u,v} \in \R^{D_Z\times D_Z}$ is a sub-matrix of $X$ such that:
$$
\tilde{X}_{u,v}(i,j) = X(i+u-1,j+v-1)
$$
with $i,j \in \{1, ..., D_Z\}$.

Please, notice that the newly obtained version of $Z^{[c]}$ might be particolarly convenient, later, to compute its gradient with respect to $b_1^{[c]}$ and $W_1^{[c]}$.

#### Second layer - Max Pooling
In the provided code, this opperation is applied on a per-channel basis with stride $s = 5$. This implies that the output $P^{[c]} \in \R^{D_P \times D_P}$, with $D_P = D_Z / s = 5$. Similarly to the previous case, one could also write that $P \in \R^{2 \times D_P \times D_P}$. The operation implemented at this step can be formalized as:
$$
P^{[c]}_{i,j} = \max(\tilde{Z}^{[c]}_{i,j})
$$

where $\tilde{Z}^{[c]}_{i,j}$ is, similarly to the previous layer, a sub-matrix of $Z^{[c]}$, such that:
$$
\tilde{Z}_{i,j}^{[c]}(u,v) = Z^{[c]}((i-1) \cdot s + u, (j-1) \cdot s+v-1)
$$
with $u,v \in \{1, ..., D_P\}$.

#### Third layer - RELU
This layer applies a simple RELU function to the output of the previous layer. The dimensions of the output from this layer match those of the previous output. Therefore: 
$$
A^{[c]} = \text{RELU}(P^{[c]})
$$
which will also be easily derivable in the backward propagation step.

#### Fourth layer - Flattening
Also this layer adds very little complexity to the network, as it only takes the $A$ output from the previous layer and lines up, sequentially, the 2 channels which were already present. It performs, therefore, a simple reshape and concatenation of the channels from the previous layer:
$$
a = \text{flatten}(A)
$$
It follows that 
$a \in R^{D_a}$, where $D_a = 2 \cdot D_P = 50$.

#### Fifth layer - Linear
From the lecture notes, we know that a linear layer simply performs a linear combination of its input with some learnable weights. In this case the layer can be written:
$$
o = W_2 a + b_2
$$
where $W_2 \in \R^{K \times D_a}$ and $b_2 \in \R^K$, with $K = 10$ being the number of categories for the problem at hand. The output is $o \in \R^K$.


#### Sixth layer
This layer applies a softmax activation function to the logits output from the previous layer. It follows, that its output $\hat{y} \in \R^K$, has the same dimensions of the input $o$. The softmax activation can be written:
$$
\hat{y}_k = \frac{\exp(o_k)}{\sum_{i=1}^K \exp(o_i)}
$$

#### Loss - Cross-entropy
Finally, the cross-entropy loss is defined as presented in the problem description for a single sample:
$$
J = CE(y,\hat{y}) = - \sum_{k=1}^K y_k \log \hat{y_k}
$$

#### Backpropagation
After having formalized the network architecture, we focus on obtaining the derivative formulae to compute the intermediate gradients in the backpropagation process, with the eventual aim of defining the gradients with respect to the parameters, in this case $W_1$, $b_1$, $W_2$, and $b_2$.

We start by writing the overall formula for backpropagation towards the latest parameters: $W_2$ and $b_2$. Using the derivative chain rule:
$$
\frac{\partial J}{\partial W_2} = \frac{\partial J}{\partial \hat{y}} \frac{\partial \hat{y}}{\partial o} \frac{\partial o}{\partial W_2}, \quad\quad
\frac{\partial J}{\partial b_2} = \frac{\partial J}{\partial \hat{y}} \frac{\partial \hat{y}}{\partial o} \frac{\partial o}{\partial b_2}
$$

We start by computing $\frac{\partial J}{\partial \hat{y}}$. which must be a vector of $K$ elements since $J$ is a scalar whereas $\hat{y_k}$ is a $K$-element vector. By deriving $J$ in each of the terms of $\hat{y}$, we obtain:

$$
\frac{\partial J}{\partial \hat{y_k}} = \frac{\partial}{\partial \hat{y_k}} \left[ - \sum_{k=1}^K y_k \log \hat{y_k}\right] = - y_k \frac{\partial}{\partial \hat{y_k}} \left[ \log \hat{y_k} \right] = - \frac{y_k}{\hat{y_k}}
$$

For the next derivative, we should derive a $K$-element vector by another $K$-element vector. This implies that the output is $\frac{\partial \hat{y}}{\partial o} \in \R^{K \times K}$. We can immediately notice that the derivation is providing different results depending on the considered index of $o$. For the sake of simplicity, we start by defining:
$$
S = \sum_{i=1}^K \exp(o_i) \implies \hat{y}_k = \frac{\exp(o_k)}{S}
$$

Now we observe:
$$
\begin{align*}
\frac{\partial \hat{y}_k}{\partial o_k} &= 
\frac{\partial}{\partial o_k} \left[ \frac{\exp(o_k)}{S} \right] = 
\frac{S \frac{\partial}{\partial o_k} \left[\exp(o_k)\right] - \exp(o_k)\frac{\partial S}{\partial o_k}}{S^2} \\
&= \frac{S \exp(o_k) - \exp(o_k)\exp(o_k)}{S^2} \\
&= \frac{\exp(o_k)}{S} - \frac{\exp(o_k)}{S} \frac{\exp(o_k)}{S} \\
&= \hat{y}_k - \hat{y}_k^2 \\
&= \hat{y}_k (1 - \hat{y}_k) \\
\end{align*}
$$

We assume now to derive the same expression, but assuming $j \neq k$:
$$
\begin{align*}
\frac{\partial \hat{y}_k}{\partial o_j} &= 
\frac{\partial}{\partial o_j} \left[ \frac{\exp(o_k)}{S} \right] = 
\frac{S \frac{\partial}{\partial o_j} \left[\exp(o_k)\right] - \exp(o_k)\frac{\partial S}{\partial o_j}}{S^2} \\
&= - \frac{\exp(o_k)\exp(o_j)}{S^2} \\
&= - \frac{\exp(o_k)}{S} \frac{\exp(o_j)}{S} \\
&= \hat{y}_k \hat{y}_j \\
\end{align*}
$$

Now we can derive $o$ with respect to $W_2$ and $b_2$:
$$
\frac{\partial o}{\partial W_2} = \frac{\partial}{\partial W_2} \left[W_2 a + b_2 \right] = a^T
$$
$$
\frac{\partial o}{\partial b_2} = \frac{\partial}{\partial b_2} \left[W_2 a + b_2 \right] = 1
$$

At this point, in order to support the following steps, we can also compute the gradient with respect to $a$.
$$
\frac{\partial o}{\partial a} = \frac{\partial}{\partial a} \left[W_2 a + b_2 \right] = W_2^T 
$$

We can proceed with the derivation of the gradients with respect to $W_1$ and $b_1$. Applying the chain rule:
$$
\frac{\partial J}{\partial W_1^{[c]}} = \frac{\partial J}{\partial o} \frac{\partial o}{\partial a} \frac{\partial a}{\partial A} \frac{\partial A}{\partial P^{[c]}} \frac{\partial P^{[c]}}{\partial Z^{[c]}} \frac{\partial Z^{[c]}}{\partial W_1^{[c]}}, \quad\quad
\frac{\partial J}{\partial b_1^{[c]}} = \frac{\partial J}{\partial o} \frac{\partial o}{\partial a} \frac{\partial a}{\partial A} \frac{\partial A}{\partial P^{[c]}} \frac{\partial P^{[c]}}{\partial Z^{[c]}} \frac{\partial Z^{[c]}}{\partial b_1^{[c]}}
$$

In this section, it is much more difficult to find a mathematic formulation of each of the derivatives in the formula. Therefore, we will proceed with a verbal description of the steps to eventually compute the target derivatives. 

Firstly, the operation that converted $A$ into $a$ was a simple flattening operation. Therefore assuming the availability of $\frac{\partial J}{\partial a} \in \R^{50}$, the derivative with respect to each element of $A$ can be obtained by simply reshaping the just mentioned derivative into the shape of $A$. This process should result in also splitting the derivative with respect to each $A^{[c]}$. Then, since the relation to obtain $A^{[c]}$ from $P^{[c]}$ was a simple RELU, the gradients with respect to each element of $P^{[c]}$ can be obtained by just applying the its derivative to the already available gradients. This consists in multiplying by 1 the gradients where the output of the RELU is bigger the 0, and by 0 all other values. Similarly, the procedure to obtain the gradient with respect to $Z^{[c]}$ consists in adjusting the shape of the gradient matrix and assigning value 1 to the elements which provided the maximum value during the forward phase, and 0 otherwise.

Finally, the gradients $Z^{[c]}$ with respect to $W_1^{[c]}$ and $b_1^{[c]}$ can be easily obtained leveraging the relation presented at the beginning:
$$
\frac{\partial Z^{[c]}}{\partial W_1^{[c]}} = \frac{\partial}{\partial W_1^{[c]}} \left[b_1^{[c]} + \sum_{u = 1}^{d} \sum_{v = 1}^{d} W_{1,u,v}^{[c]} \tilde{X}_{u,v} \right]
$$
$$
\frac{\partial Z^{[c]}}{\partial W_{1,u,v}^{[c]}} = \tilde{X}_{u,v}
$$
$$
\frac{\partial Z^{[c]}}{\partial b_1^{[c]}} = \frac{\partial}{\partial b_1^{[c]}} \left[b_1^{[c]} + \sum_{u = 1}^{d} \sum_{v = 1}^{d} W_{1,u,v}^{[c]} \tilde{X}_{u,v} \right] = 1.
$$

These computations are implemented in the dedicated functions in `p01_nn.py`.

### (b)

The full backward pass function is provided in `p01_nn.py`. The following cell allows to run the training and show the output plot. Please, notice that in order to allow running the training both in this notebook and in the original script, the script was modified to receive a dedicated path parameter.

In [ ]:
import sys
sys.path.append('src')
from src.p01_nn import main

dir_path = 'data'
main(dir_path)